In [27]:
EMA_path = "./../data/datasets/1995_pars/approved/EMA.csv"
JAPAN_path = "./../data/datasets/1995_pars/approved/PMDA.csv"
SWISSMEDIC_path = "./../data/datasets/1995_pars/approved/Swissmedic.csv"
AUSTRALIA_path = "./../data/datasets/1995_pars/approved/TGA.csv"
# FDA_path = "./../data/datasets/final_1995/FDA.csv"
# HEALTHCANADA_path = "./../data/datasets/final_1995/HealthCanada.csv"


In [28]:
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.colors as mcolors
import seaborn as sns
import json
import numpy as np
from matplotlib.ticker import FixedLocator, ScalarFormatter
import math
import os

In [29]:
# =============================
# CSV Loader
# =============================
def load_agency_csv(path: str, agency: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Spaltennamen bereinigen (falls irgendwo Leerzeichen sind)
    df.columns = df.columns.astype(str).str.strip()
    return df

# =============================
# Load all agencies into dfs
# =============================
df_ema = load_agency_csv(EMA_path, "EMA")
# df_fda = load_agency_csv(FDA_path, "FDA")
df_swissmedic = load_agency_csv(SWISSMEDIC_path, "SWISSMEDIC")
df_japan = load_agency_csv(JAPAN_path, "JAPAN")
df_australia = load_agency_csv(AUSTRALIA_path, "AUSTRALIA")
# df_healthcanada = load_agency_csv(HEALTHCANADA_path, "HEALTHCANADA")

# =============================
# Helper für Identifier-Zählung
# =============================
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def cleaned_series(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    x = x.mask(x.str.lower().isin(PLACEHOLDERS))
    return x

def agency_identifier_count(df: pd.DataFrame, agency: str) -> int:
    """
    Zählt den passenden Identifier pro Agency:
    - Japan: Anzahl Zeilen (= PDFs/Records in deinem CSV)
    - alle anderen: unique Marketing_authorisation_number (ohne Platzhalter)
    """
    if agency == "JAPAN":
        return len(df)

    s = cleaned_series(df["Marketing_authorisation_number"])
    return s.nunique(dropna=True)

# Results 1. Dataset Characteristics

Numbers per application

In [30]:
agencies = {
    # "FDA": df_fda,
    # "Health Canada": df_healthcanada,
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "Japan": df_japan,
    "Australia": df_australia,
}

# ============================================================
# 1) Record counts per agency (based on CSV rows)
# ============================================================
rows = []
total_records = sum(len(df) for df in agencies.values())

for name, df in agencies.items():
    n = len(df)
    pct = round(n / total_records * 100, 2) if total_records else 0.0
    rows.append({
        "Agency": name,
        "n_records": n,
        "%_of_overall_records": pct
    })

record_summary = pd.DataFrame(rows)

print("Record counts per agency (based on CSV rows)")
display(record_summary)

Record counts per agency (based on CSV rows)


,Agency,n_records,%_of_overall_records
0,EMA,1491,47.79
1,Swissmedic,233,7.47
2,Japan,408,13.08
3,Australia,988,31.67


#Number per Approval (Consolidated from Approved, Conditional Marketing Authorisation, Marketed)

In [31]:
# ============================================================
# Approved applications per agency (RECORD COUNTS, no normalisation)
# ============================================================

# Decisions, die als Approval zählen (exakt so geschrieben)
APPROVAL_DECISIONS = {
    "approved",
    "conditional marketing authorisation",
    "marketed",
}

rows = []
total_records_approved = 0

for name, df in agencies.items():
    # Anzahl Records insgesamt
    n_records = len(df)

    # Anzahl Records mit Approval-Decision
    n_approved = int(df["Decision"].isin(APPROVAL_DECISIONS).sum())

    rows.append({
        "Agency": name,
        "n_records_approved": n_approved,
    })

    total_records_approved += n_approved

approved_record_summary = pd.DataFrame(rows)

# Prozentanteil über alle Agencies (analog zu deinen anderen Tabellen)
approved_record_summary["%_of_overall_approved_records"] = (
    approved_record_summary["n_records_approved"] / total_records_approved * 100
).round(2)

print("Approved applications per agency (record counts, no normalisation)")
display(approved_record_summary)


Approved applications per agency (record counts, no normalisation)


,Agency,n_records_approved,%_of_overall_approved_records
0,EMA,1491,47.79
1,Swissmedic,233,7.47
2,Japan,408,13.08
3,Australia,988,31.67


Numbers per drug (unique Marketing_authorisation_number)

In [32]:
# ============================================================
# 2) Unique identifiers per agency (MA numbers; Japan = rows)
# ============================================================
def clean_ma_series(s: pd.Series) -> set:
    return set(
        s.astype("string")
         .str.strip()
         .str.lower()
         .mask(lambda x: x.isin(PLACEHOLDERS))
         .dropna()
         .unique()
    )

agency_ids = {}
overall_ids = 0

for name, df in agencies.items():
    if name == "Japan":
        n_ids = len(df)
        agency_ids[name] = n_ids
        overall_ids += n_ids
        continue

    ma_set = clean_ma_series(df["Marketing_authorisation_number"])
    n_ids = len(ma_set)
    agency_ids[name] = n_ids
    overall_ids += n_ids

rows = []
for name, n in agency_ids.items():
    pct = round(n / overall_ids * 100, 2) if overall_ids else 0.0
    rows.append({
        "Agency": name,
        "n_unique_identifiers": n,
        "%_of_overall": pct
    })

ma_summary = pd.DataFrame(rows)

print("Unique identifiers per agency (MA numbers; Japan = rows)")
display(ma_summary)

Unique identifiers per agency (MA numbers; Japan = rows)


,Agency,n_unique_identifiers,%_of_overall
0,EMA,1305,46.96
1,Swissmedic,195,7.02
2,Japan,408,14.68
3,Australia,871,31.34


# Table 1: Key characteristics of approvals

In [33]:
# ============================================================
# Constants & helpers (analysis)
# ============================================================
DRUG_CLASS_ORDER = [
    "small molecule",
    "biologics",
    "cell and gene therapy",
    "peptides and proteins",
    "vaccine",
    "not reported",
    "other",
]

def norm_series(s: pd.Series) -> pd.Series:
    """Strip whitespace, lower-case, keep NaN."""
    return s.astype("string").str.strip().str.lower()

# ============================================================
# Decision distribution (record-level, per agency)
# Output wie in deinem Screenshot (inkl. consolidated)
# ============================================================
def decision_distribution(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)

    decision = (
        df.get("Decision", pd.Series([pd.NA] * n))
          .astype("string")
          .str.strip()
          .str.lower()
    )

    # Platzhalter -> NA (damit <NA> separat erscheint)
    decision = decision.mask(decision.isin(PLACEHOLDERS), pd.NA)

    counts = decision.value_counts(dropna=False)
    dist = counts.reset_index()
    dist.columns = ["Decision", "n"]
    dist["%"] = (dist["n"] / n * 100).round(2) if n else 0.0

    consolidated_mask = decision.isin({
        "approved",
        "conditional marketing authorisation",
        "conditional marketing authorization",
    })

    consolidated_row = pd.DataFrame([{
        "Decision": "consolidated (approved + conditional marketing authorisation)",
        "n": int(consolidated_mask.sum()),
        "%": round(consolidated_mask.sum() / n * 100, 2) if n else 0.0
    }])

    return pd.concat([dist, consolidated_row], ignore_index=True)

# ============================================================
# Drug class summary (record-level)
# ============================================================
def bucket_drug_class(x) -> str:
    if pd.isna(x):
        return "not reported"

    t = str(x).strip().lower()
    if t in PLACEHOLDERS:
        return "not reported"

    if t in DRUG_CLASS_ORDER:
        return t

    return "other"

def drug_class_summary(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    s = df.get("Drug_class", pd.Series([pd.NA] * n)).map(bucket_drug_class)
    counts = s.value_counts()

    rows = []
    for c in DRUG_CLASS_ORDER:
        k = int(counts.get(c, 0))
        rows.append({
            "Drug class": c,
            "n": k,
            "%": round(k / n * 100, 2) if n else 0.0
        })
    return pd.DataFrame(rows)

# ============================================================
# Therapeutic area composition (Top 5, mention-based)
# ============================================================
def therapeutic_area_top5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    s = norm_series(df[col]).mask(lambda x: x.isin(PLACEHOLDERS)).dropna()
    if s.empty:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
         .str.lower()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    counts = exploded.value_counts().head(10)  # Top 10
    denom = int(exploded.shape[0])

    out = counts.reset_index()
    out.columns = ["Therapeutic area", "n"]
    out["%"] = (out["n"] / denom * 100).round(2) if denom else 0.0
    return out

FOCUS_DISEASE_CLASSES = [
    "diseases of the circulatory system",
    "diseases of the nervous system",
    "neoplasms",
    "endocrine, nutritional and metabolic diseases",
    "infectious diseases",
]

def therapeutic_area_focus5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )
    if s.empty:
        return pd.DataFrame(
            [{"Therapeutic area": c, "n": 0, "%": 0.0} for c in FOCUS_DISEASE_CLASSES]
        )

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
         .str.lower()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    denom = int(exploded.shape[0])  # mention-based (wie Top 5)
    counts = exploded.value_counts()

    rows = []
    for c in FOCUS_DISEASE_CLASSES:
        n = int(counts.get(c, 0))
        pct = round(n / denom * 100, 2) if denom else 0.0
        rows.append({"Therapeutic area": c, "n": n, "%": pct})

    return pd.DataFrame(rows)


# ============================================================
# Run for all agencies
# ============================================================
for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(f"{name} | Records: {len(df)}")

    print("\nDecision distribution (per application)")
    display(decision_distribution(df))

    print("\nDrug classes (applications)")
    display(drug_class_summary(df))

    print("\nTherapeutic area composition (Top 5)")
    display(therapeutic_area_top5(df))

# ============================================================
# OVERALL
# ============================================================
df_overall = pd.concat(list(agencies.values()), ignore_index=True)

print("\n" + "=" * 70)
print("OVERALL (across all agencies)")
print(f"Records: {len(df_overall)}")

print("\nDecision distribution (OVERALL)")
display(decision_distribution(df_overall))

print("\nDrug classes (OVERALL)")
display(drug_class_summary(df_overall))

print("\nTherapeutic area composition (Top 5, OVERALL)")
display(therapeutic_area_top5(df_overall))

print("\nTherapeutic area composition (selected 5 disease classes)")
display(therapeutic_area_focus5(df))



EMA | Records: 1491

Decision distribution (per application)


,Decision,n,%
0,approved,1459,97.85
1,conditional marketing authorisation,32,2.15
2,consolidated (approved + conditional marketing...,1491,100.0



Drug classes (applications)


,Drug class,n,%
0,small molecule,919,61.64
1,biologics,350,23.47
2,cell and gene therapy,27,1.81
3,peptides and proteins,89,5.97
4,vaccine,68,4.56
5,not reported,0,0.00
6,other,38,2.55



Therapeutic area composition (Top 5)


...

,Drug class,n,%
0,small molecule,1764,56.54
1,biologics,866,27.76
2,cell and gene therapy,40,1.28
3,peptides and proteins,177,5.67
4,vaccine,207,6.63
5,not reported,0,0.00
6,other,66,2.12



Therapeutic area composition (Top 5, OVERALL)


,Therapeutic area,n,%
0,neoplasms,891,20.0
1,infectious and parasitic diseases,521,11.7
2,diseases of the blood and blood-forming organs,507,11.38
3,"endocrine, nutritional, and metabolic diseases",410,9.21
4,diseases of the respiratory system,371,8.33
5,diseases of the nervous system,298,6.69
6,diseases of the musculoskeletal system and con...,259,5.81
7,diseases of the circulatory system,253,5.68
8,diseases of the digestive system,231,5.19
9,diseases of the skin,227,5.1



Therapeutic area composition (selected 5 disease classes)


,Therapeutic area,n,%
0,diseases of the circulatory system,81,5.82
1,diseases of the nervous system,80,5.75
2,neoplasms,268,19.27
3,"endocrine, nutritional and metabolic diseases",0,0.00
4,infectious diseases,0,0.00


# 4. Administration routes and pharmaceutical forms

In [34]:
def administration_route_top10(df: pd.DataFrame) -> pd.DataFrame:
    """
    Top 10 Administration routes (mention-based).
    Parenteral routes (intravenous, subcutaneous, intramuscular)
    are aggregated into a single category: 'parenteral'.
    """
    col = "Administration_route"
    if col not in df.columns:
        return pd.DataFrame(columns=["Administration route", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )

    if s.empty:
        return pd.DataFrame(columns=["Administration route", "n", "%"])

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    # ---- aggregate parenteral routes
    exploded = exploded.replace({
        "intravenous": "parenteral",
        "subcutaneous": "parenteral",
        "intramuscular": "parenteral",
    })

    counts = exploded.value_counts().head(10)
    denom = int(exploded.shape[0])

    out = counts.reset_index()
    out.columns = ["Administration route", "n"]
    out["%"] = (out["n"] / denom * 100).round(2) if denom else 0.0
    return out


def pharmaceutical_form_top10(df: pd.DataFrame) -> pd.DataFrame:
    """
    Top 10 Pharmaceutical forms (mention-based).
    Aggregations:
      - tablet + capsule -> 'tablet / capsule'
      - solution + injectable + suspension -> 'solution / injectable'
    """
    col = "Pharmaceutical_form"
    if col not in df.columns:
        return pd.DataFrame(columns=["Pharmaceutical form", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )

    if s.empty:
        return pd.DataFrame(columns=["Pharmaceutical form", "n", "%"])

    exploded = (
        s.str.split(";")
         .explode()
         .astype("string")
         .str.strip()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    # ---- aggregate pharmaceutical forms
    exploded = exploded.replace({
        "tablet": "tablet / capsule",
        "capsule": "tablet / capsule",
        "solution": "solution / injectable",
        "injectable": "solution / injectable",
    })

    counts = exploded.value_counts().head(10)
    denom = int(exploded.shape[0])

    out = counts.reset_index()
    out.columns = ["Pharmaceutical form", "n"]
    out["%"] = (out["n"] / denom * 100).round(2) if denom else 0.0
    return out

for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(f"{name}")

    print("\nAdministration route (Top 5)")
    display(administration_route_top10(df))

    print("\nPharmaceutical form (Top 5)")
    display(pharmaceutical_form_top10(df))


print("\n" + "=" * 70)
print(f"Overall")

display(administration_route_top10(df_overall))
display(pharmaceutical_form_top10(df_overall))



EMA

Administration route (Top 5)


,Administration route,n,%
0,oral,696,45.79
1,parenteral,674,44.34
2,inhalation,47,3.09
3,ocular,19,1.25
4,cutaneous,19,1.25
5,intravitreal,18,1.18
6,nasal,11,0.72
7,sublingual,4,0.26
8,intradermal,4,0.26
9,epilesional,4,0.26



Pharmaceutical form (Top 5)


,Pharmaceutical form,n,%
0,tablet / capsule,681,43.05
1,solution / injectable,323,20.42
2,powder,220,13.91
3,concentrate,124,7.84
4,suspension,57,3.6
5,solvent,43,2.72
6,lyophilisate,24,1.52
7,dispersion,19,1.2
8,eye drops,13,0.82
9,spray,12,0.76



Swissmedic

Administration route (Top 5)


...


Pharmaceutical form (Top 5)


,Pharmaceutical form,n,%
0,tablet / capsule,397,37.56
1,solution / injectable,282,26.68
2,powder,155,14.66
3,suspension,74,7.0
4,concentrate,51,4.82
5,lyophilisate,11,1.04
6,solvent,10,0.95
7,diluent,8,0.76
8,injection,8,0.76
9,spray,7,0.66



Overall


,Administration route,n,%
0,parenteral,1513,47.27
1,oral,1369,42.77
2,inhalation,84,2.62
3,intravitreal,36,1.12
4,ocular,34,1.06
5,cutaneous,27,0.84
6,topical,23,0.72
7,nasal,20,0.62
8,sublingual,14,0.44
9,vaginal,7,0.22


,Pharmaceutical form,n,%
0,tablet / capsule,1350,41.02
1,solution / injectable,718,21.82
2,powder,431,13.1
3,concentrate,211,6.41
4,suspension,154,4.68
5,injection,82,2.49
6,lyophilisate,72,2.19
7,solvent,60,1.82
8,dispersion,29,0.88
9,spray,19,0.58


# 6. Regulatory review durations

In [35]:
import pandas as pd
from IPython.display import display

# ============================================================
# Helper: compute review duration (POSITIVE ONLY)
# ============================================================
def compute_review_duration(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds 'review_duration_days' = Decision_date - Application_date.
    Keeps ONLY positive durations (> 0 days).
    """
    tmp = df.copy()

    tmp["Application_date"] = pd.to_datetime(
        tmp["Application_date"], errors="coerce", dayfirst=True
    )
    tmp["Decision_date"] = pd.to_datetime(
        tmp["Decision_date"], errors="coerce", dayfirst=True
    )

    tmp["review_duration_days"] = (
        tmp["Decision_date"] - tmp["Application_date"]
    ).dt.days

    # keep only positive durations
    tmp = tmp[tmp["review_duration_days"] > 0]

    return tmp.dropna(subset=["review_duration_days"])


# ============================================================
# Median, Q1, Q3, IQR (positive durations only)
# ============================================================
def review_duration_median_iqr(df: pd.DataFrame) -> pd.DataFrame:
    """
    Median, Q1, Q3 and IQR (Q3 - Q1) of positive review durations (days),
    stratified by abridged vs non-abridged.
    """
    tmp = compute_review_duration(df)

    tmp["Procedure"] = (
        tmp["Nonclinical_abridged"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": "Abridged", "no": "Non-abridged"})
    )

    tmp = tmp.dropna(subset=["Procedure"])

    out = (
        tmp.groupby("Procedure")["review_duration_days"]
        .agg(
            median_days="median",
            q1_days=lambda x: x.quantile(0.25),
            q3_days=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    out["iqr_days"] = out["q3_days"] - out["q1_days"]

    return out


# ============================================================
# Min / Max (positive durations only)
# ============================================================
def review_duration_min_max(df: pd.DataFrame) -> pd.DataFrame:
    """
    Min and max positive review duration (days),
    stratified by abridged vs non-abridged.
    """
    tmp = compute_review_duration(df)

    tmp["Procedure"] = (
        tmp["Nonclinical_abridged"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": "Abridged", "no": "Non-abridged"})
    )

    tmp = tmp.dropna(subset=["Procedure"])

    out = (
        tmp.groupby("Procedure")["review_duration_days"]
        .agg(
            min_days="min",
            max_days="max",
        )
        .reset_index()
    )

    return out


# ============================================================
# Agencies to include (FDA & Health Canada excluded)
# ============================================================
agencies_timeline = {
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "PMDA": df_japan,
    "TGA": df_australia,
}

# ============================================================
# Per-agency outputs
# ============================================================
for name, df in agencies_timeline.items():
    print("\n" + "=" * 70)
    print(f"{name} – Review timelines (positive durations only)")

    print("\nMedian, Q1, Q3 and IQR (days)")
    display(review_duration_median_iqr(df))

    print("\nMin / Max (days)")
    display(review_duration_min_max(df))


# ============================================================
# OVERALL (across included agencies)
# ============================================================
df_overall_timeline = pd.concat(list(agencies_timeline.values()), ignore_index=True)

print("\n" + "=" * 70)
print("OVERALL – Review timelines (positive durations only)")

print("\nMedian, Q1, Q3 and IQR (days)")
display(review_duration_median_iqr(df_overall_timeline))

print("\nMin / Max (days)")
display(review_duration_min_max(df_overall_timeline))



EMA – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


,Procedure,median_days,q1_days,q3_days,iqr_days
0,Abridged,331.0,254.0,398.0,144.0
1,Non-abridged,378.0,329.0,454.0,125.0



Min / Max (days)


,Procedure,min_days,max_days
0,Abridged,9.0,1004.0
1,Non-abridged,2.0,1627.0



Swissmedic – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


...

,Procedure,min_days,max_days
0,Abridged,4.0,2388.0
1,Non-abridged,32.0,3284.0



OVERALL – Review timelines (positive durations only)

Median, Q1, Q3 and IQR (days)


,Procedure,median_days,q1_days,q3_days,iqr_days
0,Abridged,324.0,241.5,399.50,158.00
1,Non-abridged,357.0,287.0,432.25,145.25



Min / Max (days)


,Procedure,min_days,max_days
0,Abridged,3.0,2388.0
1,Non-abridged,2.0,3284.0


# 7. Relationship between approval activity and disease incidence


In [36]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# Paths (du bist in /src, data ist eine Ebene höher)
# =========================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "Disease_burden_mapping"

GBD_PATH = DATA_DIR / "Global_disease_burden_statistics_download.csv"
MAP_PATH = DATA_DIR / "Mapping_diseases_disease_classes.csv"

# Approvals dataframe (across all agencies) – muss existieren
APPROVALS_DF = df_overall  # falls anders: hier anpassen

# =========================
# Fixed study years (GBD available up to 2021)
# =========================
START_YEAR = 1995
END_YEAR = 2021

# =========================
# Canonical disease classes (deine Liste)
# =========================
CANONICAL_CLASSES = [
    "Infectious and parasitic diseases",
    "Neoplasms",
    "Diseases of the blood and blood-forming organs",
    "Endocrine, nutritional, and metabolic diseases",
    "Mental and behavioural disorders",
    "Diseases of the nervous system",
    "Diseases of the eye and adnexa",
    "Diseases of the ear and mastoid process",
    "Diseases of the circulatory system",
    "Diseases of the respiratory system",
    "Diseases of the digestive system",
    "Diseases of the skin",
    "Diseases of the musculoskeletal system and connective tissue",
    "Diseases of the genitourinary system",
    "Pregnancy and childbirth",
    "Congenital malformations and chromosomal abnormalities",
    "Injury, poisoning and certain other consequences of external causes",
    "Other",
]

# =========================
# Helpers
# =========================
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def norm_str(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip()

def approved_mask_from_decision(decision_series: pd.Series) -> pd.Series:
    dec = decision_series.astype("string").str.strip().str.lower().fillna("")
    return dec.str.contains(r"\b(approved|authorised|authorized)\b", regex=True, na=False)

def normalise_disease_class(x) -> str:
    """
    Normalise mapping classes to CANONICAL_CLASSES.
    Fix common typos/variants and map unknowns to 'Other'.
    """
    if pd.isna(x):
        return "Other"

    t = str(x).strip()

    if t.lower() in PLACEHOLDERS:
        return "Other"

    # common typos / variants seen in your message
    fixes = {
        "Diseases of the musculskeletal system and connective tissue":
            "Diseases of the musculoskeletal system and connective tissue",
        "Diseases of the musculskeletal system and connective tissue ":
            "Diseases of the musculoskeletal system and connective tissue",
        "Congenital malformations and chromosal abnormalities":
            "Congenital malformations and chromosomal abnormalities",
        "Injury, poisining and certain other consequences of external causes":
            "Injury, poisoning and certain other consequences of external causes",
        "Injury, poisoning and certain other consequences of external causes ":
            "Injury, poisoning and certain other consequences of external causes",
        "Other ":
            "Other",
    }

    t = fixes.get(t, t)

    # if already canonical -> keep
    if t in CANONICAL_CLASSES:
        return t

    # fallback: try case-insensitive match to canonical list
    lower_map = {c.lower(): c for c in CANONICAL_CLASSES}
    if t.lower() in lower_map:
        return lower_map[t.lower()]

    return "Other"


# =========================
# 1) Load mapping file
# =========================
mapping_raw = pd.read_csv(MAP_PATH, header=None).rename(columns={0: "cause_name"})
class_cols = [c for c in mapping_raw.columns if c != "cause_name"]

mapping_long = (
    mapping_raw
    .melt(id_vars=["cause_name"], value_vars=class_cols, value_name="Disease_class_raw")
    .drop(columns=["variable"])
)

mapping_long["cause_name"] = norm_str(mapping_long["cause_name"])
mapping_long["Disease_class_raw"] = norm_str(mapping_long["Disease_class_raw"])
mapping_long = mapping_long.dropna(subset=["Disease_class_raw"])
mapping_long = mapping_long[~mapping_long["Disease_class_raw"].str.lower().isin(PLACEHOLDERS)]

# normalise to canonical classes (multi-label preserved!)
mapping_long["Disease_class"] = mapping_long["Disease_class_raw"].map(normalise_disease_class)

# drop duplicates
mapping_long = mapping_long.drop_duplicates(subset=["cause_name", "Disease_class"])

# QC: show mapping values that ended up as Other (optional)
qc_other = (
    mapping_long[mapping_long["Disease_class"] == "Other"][["Disease_class_raw"]]
    .drop_duplicates()
    .sort_values("Disease_class_raw")
)
if not qc_other.empty:
    print("\n[QC] Mapping entries that were normalised to 'Other' (check if expected):")
    display(qc_other)


# =========================
# 2) Load GBD file
# =========================
gbd = pd.read_csv(GBD_PATH)

needed_cols = {"measure_name", "metric_name", "year", "cause_name", "val"}
missing = needed_cols - set(gbd.columns)
if missing:
    raise ValueError(f"GBD file missing columns: {missing}. Found: {gbd.columns.tolist()}")

gbd["year"] = pd.to_numeric(gbd["year"], errors="coerce")
gbd["cause_name"] = norm_str(gbd["cause_name"])
gbd["measure_name"] = norm_str(gbd["measure_name"])
gbd["metric_name"] = norm_str(gbd["metric_name"])

MEASURES = ["Incidence", "Prevalence", "Deaths"]
METRIC = "Rate"  # change to "Number" if you want absolute counts

gbd_f = gbd[
    (gbd["measure_name"].isin(MEASURES)) &
    (gbd["metric_name"].str.lower() == METRIC.lower()) &
    (gbd["year"].isin([START_YEAR, END_YEAR]))
].copy()


# =========================
# 3) Map GBD causes -> Disease classes and aggregate
#    (multi-label: causes can contribute to multiple classes)
# =========================
gbd_mapped = gbd_f.merge(
    mapping_long[["cause_name", "Disease_class"]],
    on="cause_name",
    how="left"
)

# Unmapped causes -> 'Other' (so nothing gets dropped silently)
gbd_mapped["Disease_class"] = gbd_mapped["Disease_class"].fillna("Other")
gbd_mapped["Disease_class"] = gbd_mapped["Disease_class"].map(normalise_disease_class)

# Aggregate burden by Disease_class, measure_name, year
burden_class = (
    gbd_mapped.groupby(["Disease_class", "measure_name", "year"], as_index=False)["val"]
    .sum()
)

# Pivot to wide with 1995 & 2021 columns
burden_wide = (
    burden_class.pivot_table(
        index=["Disease_class", "measure_name"],
        columns="year",
        values="val",
        aggfunc="sum"
    )
    .reset_index()
    .rename(columns={
        START_YEAR: f"val_{START_YEAR}",
        END_YEAR: f"val_{END_YEAR}",
    })
)

# % change 1995 -> 2021
burden_wide["pct_change"] = (
    (burden_wide[f"val_{END_YEAR}"] - burden_wide[f"val_{START_YEAR}"]) /
    burden_wide[f"val_{START_YEAR}"] * 100
)

burden_wide["pct_change"] = burden_wide["pct_change"].mask(
    (burden_wide[f"val_{START_YEAR}"] == 0) |
    (burden_wide[f"val_{START_YEAR}"].isna()) |
    (burden_wide[f"val_{END_YEAR}"].isna()),
    np.nan
)

# Wide table with % changes per measure
burden_change_tbl = (
    burden_wide.pivot_table(
        index="Disease_class",
        columns="measure_name",
        values="pct_change",
        aggfunc="first"
    )
    .reindex(CANONICAL_CLASSES)  # keep desired order
    .reset_index()
    .rename(columns={
        "Incidence": f"Incidence_pct_change_{START_YEAR}_{END_YEAR}",
        "Prevalence": f"Prevalence_pct_change_{START_YEAR}_{END_YEAR}",
        "Deaths": f"Deaths_pct_change_{START_YEAR}_{END_YEAR}",
    })
)

# End-year absolute burden (for ranking "high burden")
burden_end_tbl = (
    burden_wide.pivot_table(
        index="Disease_class",
        columns="measure_name",
        values=f"val_{END_YEAR}",
        aggfunc="first"
    )
    .reindex(CANONICAL_CLASSES)
    .reset_index()
    .rename(columns={
        "Incidence": f"Incidence_{END_YEAR}",
        "Prevalence": f"Prevalence_{END_YEAR}",
        "Deaths": f"Deaths_{END_YEAR}",
    })
)

burden_class_summary = burden_change_tbl.merge(burden_end_tbl, on="Disease_class", how="left")


# =========================
# 4) Approvals by Disease class (approved only), per year
# =========================
appr = APPROVALS_DF.copy()
appr = appr[approved_mask_from_decision(appr["Decision"])].copy()

appr["Decision_year"] = pd.to_numeric(appr["Decision_year"], errors="coerce")
appr = appr[appr["Decision_year"].between(START_YEAR, END_YEAR, inclusive="both")]

appr["Disease_class"] = appr["Disease_class(es)"].astype("string").str.split(";")
appr = appr.explode("Disease_class")
appr["Disease_class"] = appr["Disease_class"].astype("string").str.strip()
appr = appr[appr["Disease_class"].notna() & (appr["Disease_class"] != "")]

# Normalise approvals disease classes to canonical list (unknowns -> Other)
appr["Disease_class"] = appr["Disease_class"].map(normalise_disease_class)

appr_counts = (
    appr.groupby(["Disease_class", "Decision_year"], as_index=False)
    .size()
    .rename(columns={"size": "approvals_n"})
)

appr_summary = (
    appr_counts.groupby("Disease_class", as_index=False)
    .agg(
        mean_approvals_per_year=("approvals_n", "mean"),
        max_approvals_per_year=("approvals_n", "max"),
    )
)

peak_year = (
    appr_counts.sort_values(["Disease_class", "approvals_n", "Decision_year"], ascending=[True, False, True])
    .drop_duplicates("Disease_class")[["Disease_class", "Decision_year"]]
    .rename(columns={"Decision_year": "peak_year"})
)

appr_summary = appr_summary.merge(peak_year, on="Disease_class", how="left")

# share (mentions) of approvals by disease class across all approved records
total_mentions = len(appr)
share_tbl = (
    appr["Disease_class"].value_counts()
    .reindex(CANONICAL_CLASSES, fill_value=0)
    .rename_axis("Disease_class")
    .reset_index(name="approvals_mentions_n")
)
share_tbl["approvals_mentions_pct"] = (share_tbl["approvals_mentions_n"] / total_mentions * 100).round(2) if total_mentions else 0.0


# =========================
# 5) Combine burden + approvals
# =========================
combined = (
    burden_class_summary
    .merge(appr_summary, on="Disease_class", how="left")
    .merge(share_tbl[["Disease_class", "approvals_mentions_pct"]], on="Disease_class", how="left")
)

# fill NaNs for classes with no approvals
combined["mean_approvals_per_year"] = combined["mean_approvals_per_year"].fillna(0)
combined["max_approvals_per_year"] = combined["max_approvals_per_year"].fillna(0)
combined["peak_year"] = combined["peak_year"].fillna(pd.NA)

# High burden example: top 10 by Deaths in 2021
high_burden = combined.sort_values(f"Deaths_{END_YEAR}", ascending=False).head(10)

# =========================
# 6) Outputs
# =========================
print("\n=== High-burden classes (top 10 by Deaths in 2021) ===")
display(high_burden[[
    "Disease_class",
    f"Incidence_pct_change_{START_YEAR}_{END_YEAR}",
    f"Prevalence_pct_change_{START_YEAR}_{END_YEAR}",
    f"Deaths_pct_change_{START_YEAR}_{END_YEAR}",
    "mean_approvals_per_year",
    "max_approvals_per_year",
    "peak_year",
    "approvals_mentions_pct",
]])

print("\n=== Combined table (all disease classes; canonical order) ===")
display(combined.set_index("Disease_class").reindex(CANONICAL_CLASSES).reset_index())

print("\n=== Oncology (Neoplasms) quick readout ===")
row = combined.loc[combined["Disease_class"] == "Neoplasms"]
if not row.empty:
    row = row.iloc[0]
    print(f"Neoplasms approvals share (mentions): {row['approvals_mentions_pct']}%")
    print(f"Neoplasms peak approvals/year: {row['max_approvals_per_year']} (peak year: {row['peak_year']})")
else:
    print("No 'Neoplasms' row found (check normalisation).")



[QC] Mapping entries that were normalised to 'Other' (check if expected):


,Disease_class_raw
46,"""Endocrine, nutritional, and metabolic diseases"""
71,"""Injury, poisoning and certain other consequen..."
28,Congenital malformations and chomosomal abnorm...
33,Congenital malformations and chomosomal abnorm...
39,Congenital malformations and chomosomal abnorm...
38,Congenital malformations and chomosomal abnorm...
37,Congenital malformations and chomosomal abnorm...
26,Diseases of the blood and blood-forming organs...
207,Diseases of the blood and blood-forming organs...
140,"Diseases of the circulatory system, Diseases o..."



=== High-burden classes (top 10 by Deaths in 2021) ===


/var/folders/0j/2hyszkt95x38yj11dwpbgx900000gn/T/ipykernel_31603/2393082560.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return dec.str.contains(r"\b(approved|authorised|authorized)\b", regex=True, na=False)


,Disease_class,Incidence_pct_change_1995_2021,Prevalence_pct_change_1995_2021,Deaths_pct_change_1995_2021,mean_approvals_per_year,max_approvals_per_year,peak_year,approvals_mentions_pct
17,Other,3.556633,-14.733451,-1.897817,1.555556,3,2009,0.42
8,Diseases of the circulatory system,21.007179,29.504533,7.060634,11.050000,34,2009,6.56
9,Diseases of the respiratory system,-0.859596,12.537779,1.488660,18.375000,42,2021,8.73
10,Diseases of the digestive system,-23.158407,7.970396,-47.783388,8.631579,22,2017,4.87
4,Mental and behavioural disorders,59.709815,17.051923,64.019701,5.250000,15,2015,2.49
0,Infectious and parasitic diseases,-34.070860,-15.410059,-43.313326,18.000000,51,2021,11.23
13,Diseases of the genitourinary system,0.524408,9.320193,50.349970,7.736842,15,2017,4.37
1,Neoplasms,-2.361996,3.675816,23.248351,31.333333,79,2020,19.54
5,Diseases of the nervous system,14.639111,22.669446,39.698894,10.571429,33,2021,6.59
14,Pregnancy and childbirth,-22.899602,12.014605,-59.772264,1.500000,4,2021,0.45



=== Combined table (all disease classes; canonical order) ===


,Disease_class,Deaths_pct_change_1995_2021,Incidence_pct_change_1995_2021,Prevalence_pct_change_1995_2021,Deaths_2021,Incidence_2021,Prevalence_2021,mean_approvals_per_year,max_approvals_per_year,peak_year,approvals_mentions_pct
0,Infectious and parasitic diseases,-43.313326,-34.070860,-15.410059,19.827133,3971.436485,4579.350915,18.000000,51,2021,11.23
1,Neoplasms,23.248351,-2.361996,3.675816,18.710647,1210.637248,2442.459498,31.333333,79,2020,19.54
2,Diseases of the blood and blood-forming organs,-19.657446,NaN,-14.308482,0.633637,0.000000,1632.301789,14.791667,44,2020,10.54
3,"Endocrine, nutritional, and metabolic diseases",NaN,NaN,NaN,NaN,NaN,NaN,14.863636,35,2021,9.71
4,Mental and behavioural disorders,64.019701,59.709815,17.051923,30.065360,8523.307710,17278.254065,5.250000,15,2015,2.49
5,Diseases of the nervous system,39.698894,14.639111,22.669446,9.958879,56.386111,490.710640,10.571429,33,2021,6.59
6,Diseases of the eye and adnexa,NaN,NaN,33.375479,NaN,NaN,9390.839546,4.166667,11,2015,2.23
7,Diseases of the ear and mastoid process,NaN,NaN,36.256241,NaN,NaN,20507.811907,1.000000,1,2013,0.03
8,Diseases of the circulatory system,7.060634,21.007179,29.504533,146.117452,465.536662,7293.147419,11.050000,34,2009,6.56
9,Diseases of the respiratory system,1.488660,-0.859596,12.537779,44.782317,1038.010995,7031.265365,18.375000,42,2021,8.73



=== Oncology (Neoplasms) quick readout ===
Neoplasms approvals share (mentions): 19.54%
Neoplasms peak approvals/year: 79 (peak year: 2020)


In [37]:
os.getcwd()

'/Users/jacquelinedort/Documents/DrugFork/src'

In [38]:
sorted(gbd["year"].unique())[-10:]

[np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023)]

## Overlapping Drug Names throughout agencies (case-insensitive match)

In [12]:
import pandas as pd

# =========================
# 1. Daten laden & bereinigen
# =========================

df = pd.read_csv("./../data/datasets/1995/all_decisions/Overall.csv")

df = df[
    ["Non_proprietary_name", "Drug_name", "Dataset"]
]

df["Non_proprietary_name"] = (
    df["Non_proprietary_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["Drug_name"] = (
    df["Drug_name"]
    .astype(str)
    .str.strip()
)

df = df.drop_duplicates()

# =========================
# 2. Wide table: Drug names pro Agency
# =========================

grouped = (
    df
    .groupby(["Non_proprietary_name", "Dataset"])["Drug_name"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

pivot = grouped.pivot(
    index="Non_proprietary_name",
    columns="Dataset",
    values="Drug_name"
)

pivot["n_unique_drug_names"] = pivot.apply(
    lambda row: len(set(
        name
        for cell in row.dropna()
        for name in cell
    )),
    axis=1
)

pivot.to_csv(
    "./../output/overlap/drug_names_per_Substance_per_agency.csv"
)

# =========================
# 3. Anzahl Agencies pro INN
# =========================

agency_count = (
    df
    .groupby("Non_proprietary_name")["Dataset"]
    .nunique()
    .rename("n_agencies")
    .reset_index()
)

# =========================
# 4. Drug name → Anzahl Agencies
# =========================

name_agency_map = (
    df
    .groupby(["Non_proprietary_name", "Drug_name"])["Dataset"]
    .nunique()
    .reset_index(name="n_agencies_per_name")
)

# =========================
# 5. Prüfen: global konsistenter Name?
# =========================

name_agency_map = name_agency_map.merge(
    agency_count,
    on="Non_proprietary_name",
    how="left"
)

name_agency_map["is_globally_consistent"] = (
    name_agency_map["n_agencies_per_name"]
    == name_agency_map["n_agencies"]
)

# =========================
# 6. Summary pro INN
# =========================

summary = (
    name_agency_map
    .groupby("Non_proprietary_name")
    .agg(
        n_agencies=("n_agencies", "first"),
        drug_names=("Drug_name", lambda x: sorted(set(x))),
        has_globally_consistent_name=("is_globally_consistent", "any")
    )
    .reset_index()
)

summary.to_csv(
    "./../output/overlap/drug_name_consistency_summary.csv",
    index=False
)

# =========================
# 7. Zusatz-Output: detailliertes Mapping
# =========================

name_agency_map.to_csv(
    "./../output/overlap/drug_name_origin_by_agency_count.csv",
    index=False
)

# Absolute Zahlen
n_total = summary.shape[0]
n_consistent = summary["has_globally_consistent_name"].sum()

# Prozent
percent_consistent = (n_consistent / n_total) * 100

print(f"Total number of substances (INNs): {n_total}")
print(f"Substances with ≥1 globally consistent drug name: {n_consistent}")
print(f"Proportion with globally consistent name: {percent_consistent:.1f}%")




Total number of substances (INNs): 4655
Substances with ≥1 globally consistent drug name: 4075
Proportion with globally consistent name: 87.5%


Anzahl Substanzen mit mehr als einem Produktnamen

In [2]:
(pivot["n_unique_drug_names"] > 1).mean()


np.float64(0.3726627981947131)

In [8]:
import json
import re
import pandas as pd

input_path = "./../data/FDA/with_extracted_data_disease_class/FDA.json"
output_csv = "./../output/Disease_names/FDA_extracted_disease_names.csv"

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

entries = data.values() if isinstance(data, dict) else data

diseases = set()

for entry in entries:
    value = entry.get("Indications_and_usage_disease_name_extracted")
    if not value or not isinstance(value, str):
        continue

    # 1) bevorzugt: Inhalte in <...>
    angle_matches = re.findall(r"<([^>]+)>", value)

    if angle_matches:
        for d in angle_matches:
            d = d.strip()
            if d:
                diseases.add(d)
        continue

    # 2) fallback: falls doch mal [ ... ] vorkommt
    bracket_matches = re.findall(r"\[([^\]]+)\]", value)
    for match in bracket_matches:
        # falls mehrere Krankheiten in einer Klammer stehen
        parts = re.split(r";|,", match)
        for part in parts:
            d = part.strip()
            if d:
                diseases.add(d)

# als sortierte Liste
disease_list = sorted(diseases)

print(f"Number of unique disease names: {len(disease_list)}")
print("Preview:", disease_list[:20])

# speichern
df_out = pd.DataFrame(disease_list, columns=["Disease_name"])
df_out.to_csv(output_csv, index=False)

print(f"Saved to: {output_csv}")



Number of unique disease names: 1355
Preview: ['AIDS-Related Kaposi Sarcoma', 'Abnormal Uterine Bleeding', 'Abnormalities, Liver', 'Abruptio Placentae', 'Abscess', 'Absence Epilepsy', 'Absence Seizures', 'Acetaminophen Overdose', 'Achondroplasia', 'Acinetobacter Infections', 'Acne', 'Acne Rosacea', 'Acne Vulgaris', 'Acquired Immunodeficiency Syndrome', 'Acromegaly', 'Actinic Keratosis', 'Actinomycosis', 'Active Tuberculosis', 'Acute Bacterial Otitis Media', 'Acute Bacterial Skin and Skin Structure Infections']
Saved to: ./../output/Disease_names/FDA_extracted_disease_names.csv
